In [1]:
import pandas as pd
import numpy as np

df_test = pd.read_csv('../data/test.csv')
df_train = pd.read_csv('../data/train.csv')

df_summary = pd.DataFrame({'dtype': df_train.dtypes, 'nulls': df_train.isna().sum(), 'nununique': df_train.nunique(), 'sample_1': df_train.iloc[0], 'sample_2': df_train.iloc[1], 'sample_3': df_train.iloc[2]})
df_summary



,dtype,nulls,nununique,sample_1,sample_2,sample_3
Id,int64,0,1460,1,2,3
MSSubClass,int64,0,15,60,20,60
MSZoning,str,0,5,RL,RL,RL
LotFrontage,float64,259,110,65.0,80.0,68.0
LotArea,int64,0,1073,8450,9600,11250
...,...,...,...,...,...,...
MoSold,int64,0,12,2,5,9
YrSold,int64,0,5,2008,2007,2008
SaleType,str,0,9,WD,WD,WD
SaleCondition,str,0,6,Normal,Normal,Normal


In [2]:
num_cols = df_train.select_dtypes(include='number').columns.tolist()

cat_cols = df_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

ord_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageCond', 'GarageQual', 'PoolQC', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']

cat_cols = [col for col in cat_cols if col not in ord_cols]

ord_cols = [col for col in ord_cols if col in df_train.columns]

num_cols = [col for col in num_cols if col != 'SalePrice' and col != 'Id']

C:\Users\yazid\AppData\Local\Temp\ipykernel_12720\2204705351.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()


In [3]:

quality_order = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
quality_order_none = ['None'] + quality_order
exposure_none_order = ['None', 'No', 'Mn', 'Av', 'Gd']
finish_none_order = ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
pool_none_order = ['None', 'Fa', 'Gd', 'Ex']

regular_ord_cols = ['ExterQual', 'ExterCond', 'HeatingQC', 'KitchenQual']
none_ord_cols = ['BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'BsmtQual', 'BsmtCond', 'GarageCond', 'GarageQual', 'PoolQC', 'FireplaceQu']
print(cat_cols)
none_cat_cols = ['Alley', 'MasVnrType', 'GarageType', 'GarageFinish', 'Fence', 'MiscFeature']
regular_cat_cols = [col for col in cat_cols if col not in none_cat_cols]


['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'Heating', 'CentralAir', 'Electrical', 'Functional', 'GarageType', 'GarageFinish', 'PavedDrive', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition']


In [4]:

ordinal_categories = {
    'ExterQual': quality_order,
    'ExterCond': quality_order,
    'BsmtQual': quality_order_none,
    'BsmtCond': quality_order_none,
    'HeatingQC': quality_order,
    'KitchenQual': quality_order,
    'FireplaceQu': quality_order_none,
    'GarageCond': quality_order_none,
    'GarageQual': quality_order_none,
    'PoolQC': pool_none_order,
    'BsmtExposure': exposure_none_order,
    'BsmtFinType1': finish_none_order,
    'BsmtFinType2': finish_none_order
}

regular_categories = [ordinal_categories[col] for col in regular_ord_cols]
none_categories = [ordinal_categories[col] for col in none_ord_cols]



In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV

num_pipeline = Pipeline(steps=[('impute', SimpleImputer(strategy='median'))])
regular_cat_pipeline = Pipeline(steps=[('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))])
none_cat_pipeline = Pipeline(steps=[('impute', SimpleImputer(strategy='constant', fill_value='None')), ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))])
regular_ord_pipeline = Pipeline(steps=[('impute', SimpleImputer(strategy='most_frequent')), ('ordinal', OrdinalEncoder(categories=regular_categories, handle_unknown='use_encoded_value', unknown_value=-1))])
none_ord_pipeline = Pipeline(steps=[('impute', SimpleImputer(strategy='constant', fill_value='None')), ('ordinal', OrdinalEncoder(categories=none_categories, handle_unknown='use_encoded_value', unknown_value=-1))])

param_distributions = {
    'model__n_estimators': [100, 300, 600, 1000],
    'model__learning_rate': [0.01, 0.03, 0.05, 0.1],
    'model__max_depth': [2,3,4,5,6],
    'model__min_child_weight': [1,3,5],
    'model__subsample': [0.8,0.9,1.0]
}
preprocessor = ColumnTransformer(transformers=[('num', num_pipeline, num_cols), ('regular_cat', regular_cat_pipeline, regular_cat_cols), ('none_cat', none_cat_pipeline, none_cat_cols), ('regular_ord', regular_ord_pipeline, regular_ord_cols), ('none_ord', none_ord_pipeline, none_ord_cols)])

model1 = XGBRegressor(learning_rate=0.01, n_estimators=100, random_state=42, max_depth=6)

full_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model1)])

X_train = df_train.drop(columns=['SalePrice', 'Id'])
X_test = df_test.drop(columns=['Id'])
y_train = np.log1p(df_train['SalePrice'])
test_id = df_test['Id']

In [6]:
from sklearn.dummy import DummyRegressor
from xgboost import cv
from sklearn.model_selection import cross_val_score, KFold

cv = KFold(n_splits=5, shuffle=True, random_state=42)

baseline = DummyRegressor(strategy='mean')

baseline_scores = cross_val_score(
    baseline, 
    X_train,
    y_train,
    cv=cv,
    scoring='neg_mean_squared_error'
)

baseline_rmsle = np.sqrt(-baseline_scores.mean())

In [7]:

msle = cross_val_score(full_pipeline, X_train, y_train, cv=cv, scoring='neg_mean_squared_error')
rmsle = np.sqrt(-msle.mean())

print(baseline_rmsle, rmsle)

0.3995729236866519 0.21944881673891328


In [8]:
search = RandomizedSearchCV(
    estimator=full_pipeline,
    param_distributions=param_distributions,
    n_iter=30,
    scoring='neg_mean_squared_error',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    return_train_score=True,
    refit=False,
    error_score='raise'
)

search.fit(X_train, y_train)

results = pd.DataFrame(search.cv_results_)

In [9]:
results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__subsample,param_model__n_estimators,param_model__min_child_weight,param_model__max_depth,param_model__learning_rate,params,...,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,mean_train_score,std_train_score
0,1.959732,0.033013,0.051671,0.005679,0.9,300,3,6,0.03,"{'model__subsample': 0.9, 'model__n_estimators...",...,-0.018158,0.006168,13,-9.729014e-04,-1.028899e-03,-9.742779e-04,-1.027034e-03,-9.625301e-04,-9.931284e-04,2.873963e-05
1,0.880421,0.070541,0.061283,0.004143,1.0,100,1,5,0.03,"{'model__subsample': 1.0, 'model__n_estimators...",...,-0.021746,0.004603,23,-6.579574e-03,-6.478738e-03,-6.509426e-03,-6.293258e-03,-6.826157e-03,-6.537431e-03,1.726198e-04
2,1.310864,0.184855,0.051964,0.009285,0.8,600,3,3,0.01,"{'model__subsample': 0.8, 'model__n_estimators...",...,-0.018436,0.005157,16,-8.842379e-03,-9.084484e-03,-8.047052e-03,-8.863919e-03,-9.150635e-03,-8.797694e-03,3.941221e-04
3,0.792876,0.020449,0.047119,0.005197,0.8,600,3,2,0.03,"{'model__subsample': 0.8, 'model__n_estimators...",...,-0.017431,0.005357,9,-7.874677e-03,-7.920652e-03,-7.207606e-03,-7.865440e-03,-8.211669e-03,-7.816009e-03,3.297079e-04
4,2.415880,0.074690,0.075970,0.012015,0.8,1000,3,4,0.05,"{'model__subsample': 0.8, 'model__n_estimators...",...,-0.017277,0.005634,7,-2.026811e-04,-2.390761e-04,-2.195414e-04,-2.566338e-04,-2.310596e-04,-2.297984e-04,1.815685e-05
5,0.621754,0.016397,0.049041,0.004267,1.0,300,5,3,0.01,"{'model__subsample': 1.0, 'model__n_estimators...",...,-0.023461,0.004490,26,-1.580214e-02,-1.605623e-02,-1.444318e-02,-1.519423e-02,-1.625837e-02,-1.555083e-02,6.592817e-04
6,0.671919,0.025758,0.048930,0.010170,0.8,300,5,3,0.01,"{'model__subsample': 0.8, 'model__n_estimators...",...,-0.022851,0.004451,24,-1.565983e-02,-1.607550e-02,-1.443301e-02,-1.535714e-02,-1.632535e-02,-1.557017e-02,6.591210e-04
7,0.990133,0.036796,0.063846,0.006854,0.9,100,1,6,0.01,"{'model__subsample': 0.9, 'model__n_estimators...",...,-0.047214,0.006299,28,-3.278999e-02,-3.466528e-02,-3.514739e-02,-3.317582e-02,-3.562067e-02,-3.427983e-02,1.107929e-03
8,1.422835,0.047961,0.057605,0.006848,0.9,300,1,5,0.03,"{'model__subsample': 0.9, 'model__n_estimators...",...,-0.017993,0.005057,12,-1.870611e-03,-1.933086e-03,-1.852426e-03,-1.857230e-03,-1.930316e-03,-1.888734e-03,3.559595e-05
9,0.870084,0.033688,0.054840,0.006246,0.8,600,5,2,0.05,"{'model__subsample': 0.8, 'model__n_estimators...",...,-0.017109,0.005213,4,-6.269573e-03,-6.271512e-03,-5.675463e-03,-6.555010e-03,-6.349600e-03,-6.224231e-03,2.934518e-04


In [10]:
results['train_rmsle'] = np.sqrt(-results['mean_train_score'])

results['validation_rmsle'] = np.sqrt(
    -results['mean_test_score']
)

results['overfit_gap'] = (
    results['validation_rmsle'] - results['train_rmsle']
)

max_gap = 0.05

accepted_results = results[results['overfit_gap'] <= max_gap]

best_index = accepted_results['validation_rmsle'].idxmin()

best_params = results.loc[best_index, 'params']

In [11]:
from sklearn.base import clone

final_pipeline = clone(full_pipeline)
final_pipeline.set_params(**best_params)
final_pipeline.fit(X_train, y_train)

selected_scores = -cross_val_score(
    final_pipeline, 
    X_train, 
    y_train,
    cv=cv,
    scoring='neg_mean_squared_error'
)

print("Selected RMSLE:", np.sqrt(selected_scores.mean()))

Selected RMSLE: 0.13157129038955456
